# Cross-Trait Genetic Correlation

Goal: estimate genetic correlation for two or more traits from munged summary statistics and one matched LD-score reference.

This notebook uses a tiny synthetic dataset so every cell can run as a smoke test. Replace the toy tables with real `sumstats.parquet` files plus their `sumstats.metadata.json` sidecars, or explicit legacy `.sumstats.gz` compatibility files, and a canonical LD-score result directory containing `manifest.json` and `ldscore.baseline.parquet`. Current LD-score directories keep `ldscore.baseline.parquet` as one flat file with chromosome-aligned row groups listed in `manifest.json`.

Output directories are created when missing and reused when present. Existing fixed workflow files such as `rg.tsv`, `rg_full.tsv`, `h2_per_trait.tsv`, optional `pairs/`, and `rg.log` are refused before writing unless you pass `--overwrite` or `overwrite=True`. Logs are audit files and are not returned as scientific output paths.


In [ ]:
from pathlib import Path
import sys

def find_src_dir(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "ldsc").exists():
            return candidate / "src"
        nested = candidate / "ldsc_py3_restructured" / "src"
        if (nested / "ldsc").exists():
            return nested
    raise FileNotFoundError("Could not find src/ldsc from the current working directory.")

SRC_DIR = find_src_dir(Path.cwd())
REPO_DIR = SRC_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(SRC_DIR)

In [ ]:
import os
import subprocess
import tempfile

import numpy as np
import pandas as pd

from ldsc import (
    GlobalConfig,
    LDScoreDirectoryWriter,
    LDScoreOutputConfig,
    LDScoreResult,
    RegressionConfig,
    RegressionRunner,
    SumstatsMunger,
    set_global_config,
)
from ldsc.sumstats_munger import SumstatsTable


## Python API

The public API consumes two or more `SumstatsTable` objects and one `LDScoreResult`. In a real run, create the sumstats with `SumstatsMunger().run(...)` or `load_sumstats(...)`; current disk artifacts reload their `sumstats.metadata.json` sidecars so config provenance is retained. Create LD scores with `run_ldscore(...)` or `load_ldscore_from_dir(...)`.


In [ ]:
GLOBAL_CONFIG = GlobalConfig(snp_identifier="rsid", genome_build="hg38")
set_global_config(GLOBAL_CONFIG)

n_snps = 20
snps = [f"rs{i}" for i in range(1, n_snps + 1)]
index = np.arange(n_snps)

trait_1 = SumstatsTable(
    data=pd.DataFrame(
        {
            "SNP": snps,
            "CHR": ["1"] * n_snps,
            "POS": np.arange(100, 100 + n_snps),
            "Z": np.linspace(-2.0, 2.0, n_snps) + 0.1 * np.sin(index),
            "N": np.full(n_snps, 10000.0),
            "A1": ["A"] * n_snps,
            "A2": ["G"] * n_snps,
        }
    ),
    has_alleles=True,
    source_path="synthetic_trait_1",
    trait_name="trait_1",
    provenance={},
    config_snapshot=GLOBAL_CONFIG,
)

trait_2 = SumstatsTable(
    data=pd.DataFrame(
        {
            "SNP": snps,
            "CHR": ["1"] * n_snps,
            "POS": np.arange(100, 100 + n_snps),
            "Z": np.linspace(-1.5, 1.8, n_snps) + 0.2 * np.cos(index),
            "N": np.full(n_snps, 12000.0),
            "A1": ["A"] * n_snps,
            "A2": ["G"] * n_snps,
        }
    ),
    has_alleles=True,
    source_path="synthetic_trait_2",
    trait_name="trait_2",
    provenance={},
    config_snapshot=GLOBAL_CONFIG,
)

ldscore_result = LDScoreResult(
    baseline_table=pd.DataFrame(
        {
            "CHR": ["1"] * n_snps,
            "SNP": snps,
            "POS": np.arange(100, 100 + n_snps),
            "regression_ld_scores": np.linspace(1.1, 1.5, n_snps),
            "base": np.linspace(1.0, 2.0, n_snps),
        }
    ),
    query_table=None,
    count_records=[
        {
            "group": "baseline",
            "column": "base",
            "all_reference_snp_count": 120.0,
            "common_reference_snp_count": 100.0,
        }
    ],
    baseline_columns=["base"],
    query_columns=[],
    ld_reference_snps=frozenset(snps),
    ld_regression_snps=frozenset(snps),
    chromosome_results=[],
    config_snapshot=GLOBAL_CONFIG,
)

trait_1.data.head()

In [ ]:
runner = RegressionRunner(
    global_config=GLOBAL_CONFIG,
    regression_config=RegressionConfig(n_blocks=10, use_intercept=False),
)
result = runner.estimate_rg_pairs([trait_1, trait_2], ldscore_result)
api_summary = result.rg
api_summary

## CLI

For real inputs, run `ldsc munge-sumstats` once per trait, then run `ldsc rg --sumstats-sources ... --ldscore-dir ... --output-dir ...`. Munge output directories reject any owned `sumstats.*` sibling unless `--overwrite` or `overwrite=True` is set; successful overwrites remove stale unselected sumstats formats. Add `--overwrite` to `rg` only for intentional reruns that replace existing fixed files, including `rg.log`. Use `--anchor-trait-file` to compute only anchor-vs-rest pairs. The cell below writes the toy tables to temporary LDSC artifacts and runs the same CLI path.


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir_raw:
    tmpdir = Path(tmpdir_raw)
    trait_1_dir = tmpdir / "trait_1"
    trait_2_dir = tmpdir / "trait_2"
    ldscore_dir = tmpdir / "baseline_ldscores"
    output_dir = tmpdir / "trait_1_trait_2"

    trait_1_path = Path(SumstatsMunger().write_output(trait_1, trait_1_dir, overwrite=True))
    trait_2_path = Path(SumstatsMunger().write_output(trait_2, trait_2_dir, overwrite=True))
    LDScoreDirectoryWriter().write(
        ldscore_result,
        LDScoreOutputConfig(output_dir=ldscore_dir, overwrite=True),
    )

    env = os.environ.copy()
    env["PYTHONPATH"] = str(SRC_DIR)
    command = [
        sys.executable,
        "-m",
        "ldsc",
        "rg",
        "--sumstats-sources",
        str(trait_1_path),
        str(trait_2_path),
        "--ldscore-dir",
        str(ldscore_dir),
        "--count-kind",
        "common",
        "--n-blocks",
        "10",
        "--no-intercept",
        "--output-dir",
        str(output_dir),
    ]
    completed = subprocess.run(command, cwd=REPO_DIR, env=env, capture_output=True, text=True)
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError(f"ldsc rg failed with exit code {completed.returncode}")
    cli_summary = pd.read_csv(output_dir / "rg.tsv", sep="\t")

cli_summary
